# Experiments
This file contains the experiments that were used to find the optimal configuration of our LSTM model. The code of the model itself is located in [solution.py](solution.py).

Imports load_raw_data and inspect_dataset from solution.py and
prints the dataset summary. Run this to confirm the data is readable
and to learn input_size before building the model.

In [3]:
%pip install --upgrade pip
%pip uninstall -y torch torchvision torchaudio
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Found existing installation: torch 2.5.1+cu121
Uninstalling torch-2.5.1+cu121:
  Successfully uninstalled torch-2.5.1+cu121
Found existing installation: torchvision 0.20.1+cu121
Uninstalling torchvision-0.20.1+cu121:
  Successfully uninstalled torchvision-0.20.1+cu121
Found existing installation: torchaudio 2.5.1+cu121
Uninstalling torchaudio-2.5.1+cu121:
  Successfully uninstalled torchaudio-2.5.1+cu121
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.


Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached torch-2.5.1%2Bcu121-cp312-cp312-win_amd64.whl (2449.3 MB)
  Using cached torchvision-0.20.1%2Bcu121-cp312-cp312-win_amd64.whl (6.1 MB)
  Using cached torchaudio-2.5.1%2Bcu121-cp312-cp312-win_amd64.whl (4.1 MB)

   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ---------------------------------------- 0/3 [torch]
   ------------------

In [10]:
import sys, torch
print(torch.__version__, )
print(f"Using device: {torch.device('cuda') if torch.cuda.is_available() else 'cpu'}, {torch.cuda.get_device_name(0) if torch.cuda.is_available() else ' --- '}")

2.5.1+cu121
Using device: cuda, NVIDIA GeForce RTX 3060 Ti


# Dataset

In [2]:
from solution import load_raw_data, inspect_dataset, TRAINING_DATA_PATH

training_data = load_raw_data(TRAINING_DATA_PATH)
inspect_dataset(training_data)

Samples             : 2939
Input size          : 1  (features per time step)
Min sequence length : 4
Max sequence length : 6308
First sequence      : 
	shape - (4756,) 
	label nr - 0 
	composer - bach
Composer class counts:
    0 (bach): 1630 samples
    1 (beethoven): 478 samples
    2 (debussy): 154 samples
    3 (scarlatti): 441 samples
    4 (victoria): 236 samples


Since the sequences can have various lengths (between 4 and 6308!) we need to to make their shape the same.

In [3]:
from torch.utils.data import DataLoader
from solution import ChordDataset, pad_batch


dataset = ChordDataset(training_data)
loader = DataLoader(
        dataset,
        batch_size=len(training_data),
        shuffle=True,
        collate_fn=pad_batch,
)

padded_sequences, labels, original_lengths = next(iter(loader))

print(f"Padded batch shape     : {padded_sequences.shape}")
print(f"First sequence (padded): {padded_sequences[0].shape}")
print(f"Original lengths       : {original_lengths[:30]}")

Padded batch shape     : torch.Size([2939, 6308, 1])
First sequence (padded): torch.Size([6308, 1])
Original lengths       : [140, 339, 528, 1740, 300, 288, 552, 68, 596, 633, 180, 192, 128, 270, 68, 312, 602, 112, 68, 116, 198, 228, 1061, 138, 270, 342, 224, 412, 172, 375]


# Experiment evaluation helpers

In [4]:
import copy
import numpy as np
from sklearn.model_selection import StratifiedKFold
from solution import NetConfig, LSTMTrainer, TRAINING_DATA_PATH


def calc_balanced_accuracy(predictions: np.ndarray, targets: np.ndarray) -> float:
    """Mean per-class recall (balanced accuracy)."""
    return float(np.mean([
        (predictions[targets == class_id] == class_id).mean()
        for class_id in range(5)
        if (targets == class_id).sum() > 0
    ]))


def evaluate_cv(
    experiment_name: str,
    config: NetConfig,
    data: list,
    num_folds: int = 5,
) -> None:
    """Train and evaluate a single NetConfig using StratifiedKFold."""
    all_labels = np.array([label for _, label in data])
    fold_splitter = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)
    fold_scores = []

    for train_indices, val_indices in fold_splitter.split(np.zeros(len(all_labels)), all_labels):
        print(".", end=" ")
        
        train_fold = [data[i] for i in train_indices]
        val_fold   = [data[i] for i in val_indices]
        val_labels = all_labels[val_indices]

        fold_trainer = LSTMTrainer(copy.deepcopy(config))
        fold_trainer.fit(train_fold)
        fold_predictions = fold_trainer.predict(val_fold)

        fold_scores.append(calc_balanced_accuracy(fold_predictions, val_labels))

    mean_score, std_score = np.mean(fold_scores), np.std(fold_scores)
    print(f"{experiment_name:50s}  acc = {mean_score:.4f} ± {std_score:.4f}")

# Experiments

### Hidden size

In [ ]:
for hidden_size in [256, 512, 1024]:
    config = NetConfig(hidden_size=hidden_size, num_layers=2, dropout=0.2, epochs=15, bidirectional=True)
    evaluate_cv(f"hidden={hidden_size} layers=2 dropout=0.2", config, training_data)    

. . . . . hidden=32 layers=2 dropout=0.3                      **acc = 0.4270 ± 0.0736** <br/>
. . . . . hidden=64 layers=2 dropout=0.3                      **acc = 0.5412 ± 0.0480** <br/>
. . . . . hidden=128 layers=2 dropout=0.3                     **acc = 0.5665 ± 0.0329** <br/>
. . . . . hidden=256 layers=2 dropout=0.3                     **acc = 0.6104 ± 0.0192** <br/>

. . . . . hidden=256 layers=2 dropout=0.2                     **acc = 0.6112 ± 0.0310** <br/>
. . . . . hidden=512 layers=2 dropout=0.2                     **acc = 0.6504 ± 0.0100** <br/>

### Number of layers

In [ ]:
for num_layers in [4]:
    # config = NetConfig(hidden_size=256, num_layers=num_layers, dropout=0.2, epochs=15, bidirectional=True)
    # evaluate_cv(f"hidden=256 layers={num_layers} dropout=0.2", config, training_data)
    
    config = NetConfig(hidden_size=1024, num_layers=num_layers, dropout=0.2, epochs=15, bidirectional=True)
    evaluate_cv(f"hidden=1024 layers={num_layers} dropout=0.2", config, training_data)

. 

. . . . . hidden=64 layers=1 dropout=0.3                      **acc = 0.4747 ± 0.0269** <br/>
. . . . . hidden=64 layers=2 dropout=0.3                      **acc = 0.4551 ± 0.0503** <br/>
. . . . . hidden=64 layers=3 dropout=0.3                      **acc = 0.4730 ± 0.0308** <br/>

. . . . . hidden=256 layers=1 dropout=0.2                     **acc = 0.5749 ± 0.0530** <br/>
. . . . . hidden=256 layers=2 dropout=0.2                     **acc = 0.5936 ± 0.0256** <br/>
. . . . . hidden=256 layers=3 dropout=0.2                     **acc = 0.5574 ± 0.0202** <br/>
. . . . . hidden=256 layers=4 dropout=0.2                     **acc = 0.5176 ± 0.0208**<br/>

. . . . . hidden=1024 layers=1 dropout=0.2                    **acc = 0.5574 ± 0.0300** <br/>
. . . . . hidden=1024 layers=2 dropout=0.2                    **acc = 0.6294 ± 0.0382** <br/>

### Dropout

In [7]:
for dropout in [0.0, 0.2, 0.3, 0.5]:
    config = NetConfig( hidden_size=256, num_layers=2, dropout=dropout, epochs=15, bidirectional=False)
    evaluate_cv(f"hidden=256 layers=2 dropout={dropout}", config, training_data)

. . . . . hidden=256 layers=2 dropout=0.0                     acc = 0.5767 ± 0.0295
. . . . . hidden=256 layers=2 dropout=0.2                     acc = 0.5292 ± 0.0357
. . . . . hidden=256 layers=2 dropout=0.3                     acc = 0.5303 ± 0.0438
. . . . . hidden=256 layers=2 dropout=0.5                     acc = 0.5302 ± 0.0339


. . . . . hidden=256 layers=2 dropout=0.0                     **acc = 0.5767 ± 0.0295** <br/>
. . . . . hidden=256 layers=2 dropout=0.2                     **acc = 0.5292 ± 0.0357** <br/>
. . . . . hidden=256 layers=2 dropout=0.3                     **acc = 0.5303 ± 0.0438** <br/>
. . . . . hidden=256 layers=2 dropout=0.5                     **acc = 0.5302 ± 0.0339** <br/>

### Bidirectionality

In [7]:
for bidir in [False, True]:
    config = NetConfig( hidden_size=256, num_layers=2, dropout=0.3, epochs=15, bidirectional=bidir )
    evaluate_cv(f"hidden=256 layers=2 dropout=0.3 bidir={bidir}", config, training_data)

. . . . . hidden=256 layers=2 dropout=0.3 bidir=False         acc = 0.5508 ± 0.0247
. . . . . hidden=256 layers=2 dropout=0.3 bidir=True          acc = 0.5907 ± 0.0271


. . . . . hidden=64 layers=2 dropout=0.3 bidir=False          **acc = 0.4816 ± 0.0320** <br/>
. . . . . hidden=64 layers=2 dropout=0.3 bidir=True           **acc = 0.5258 ± 0.0430** <br/>

. . . . . hidden=256 layers=2 dropout=0.3 bidir=False         acc = 0.5508 ± 0.0247
. . . . . hidden=256 layers=2 dropout=0.3 bidir=True          acc = 0.5907 ± 0.0271

### Class weights and sampling experiments

Since the dataset is quite unbalanced, it could be beneficial to consider using a method to counter that. Here we will be comparing 3 methods and a base model to see which performs best.

In [8]:
configs = [
    (   "base",
        NetConfig(hidden_size=256,num_layers=2,dropout=0.3,epochs=15,bidirectional=False,batch_size=32)),
    
    (   "class_weights",
        NetConfig(hidden_size=256,num_layers=2,dropout=0.3,epochs=15,bidirectional=False,batch_size=32,use_class_weights=True)),
    
    (   "oversample",
        NetConfig( hidden_size=256, num_layers=2, dropout=0.3, epochs=15, bidirectional=False, batch_size=32, balance_strategy="oversample" )),
        
    (  "undersample",
        NetConfig(hidden_size=256,num_layers=2,dropout=0.3,epochs=15,bidirectional=False,batch_size=32,balance_strategy="undersample")),
]

for name, config in configs:
    evaluate_cv(f"experiment={name}", config, training_data)

. . . . . experiment=base                                     acc = 0.5545 ± 0.0424
. . . . . experiment=class_weights                            acc = 0.5891 ± 0.0188
. . . . . experiment=oversample                               acc = 0.5708 ± 0.0518
. . . . . experiment=undersample                              acc = 0.4939 ± 0.0287


. . . . . experiment=base                                     **acc = 0.4712 ± 0.0272** <br/>
. . . . . experiment=class_weights                            **acc = 0.5343 ± 0.0254** <br/>
. . . . . experiment=oversample                               **acc = 0.5615 ± 0.0236** <br/>
. . . . . experiment=undersample                              **acc = 0.4619 ± 0.0163** <br/>

In [ ]:
# TODO: sprawdź attention=True/False
# TODO: sprawdź scheduler config
# TODO: grid search parametrów
# TODO: najlepszą konfigurację wykorzystać (tu lub w funkcji main solution.py) do wygenerowania finalnych predykcji na danych testowych. 

# Jakby coś to w tych wcześniejszyvh eksperymentach mamy duże cross validatopn bo 5, wiec jesli nie bedziesz miec cierpliwosci to mozesz je zmniejszyc, do min 3

# Wydaje mi sie ze dobre wyniiki beda z hidden_size=1024, ale straszliwie wydłuża to czas treningu. 

# Mamy zaimplementowane early stopping, wiec jak znajdziesz ta finalna konfiguracje to mozesz spokojnie dac duzo epok, zeby sie upewnic ze model sie wytrenowal

# z racji tego, że treningi z cv troche zajmuja to polecam dobrze przemyśleć testowane konfiguracje

### Attention

In [9]:
for attention in [False, True]:
    config = NetConfig( hidden_size=256, num_layers=2, dropout=0.3, epochs=15, bidirectional=False, batch_size=32, balance_strategy="oversample", attention=attention )
    evaluate_cv(f"attention={attention}", config, training_data)

. . . . . attention=False                                     acc = 0.6303 ± 0.0240
. . . . . attention=True                                      acc = 0.6257 ± 0.0307


Attention parameter on provides less accuracy than off. We will use no attention parameter in next experiments.

### Scheduler

In [5]:
schedulers = [None, "plateau", "cosine"]
for scheduler in schedulers:
    config = NetConfig( hidden_size=256, num_layers=2, dropout=0.3, epochs=15, bidirectional=False, batch_size=32, balance_strategy="oversample", attention=False,
        scheduler_type=scheduler)
    evaluate_cv(f"scheduler={scheduler}", config, training_data)

. . . . . scheduler=None                                      acc = 0.6569 ± 0.0251
. . . . . scheduler=plateau                                   acc = 0.6251 ± 0.0092
. . . . . scheduler=cosine                                    acc = 0.6205 ± 0.0081


Experiments have shown that, the model achieves the highest accuracy with scheduler_type set to None. Other schedulers were decreasing the learning rate too soon, which ended up with sticking to local minimums and not learnign any more.

### Grid Search

In [11]:
import copy
import itertools

baseline_config = NetConfig(
    num_layers=2, 
    dropout=0.3, 
    epochs=15, 
    batch_size=32, 
    scheduler_type=None,
    use_batch_norm=False,
    use_class_weights=False
)

grid_parameters = {
    "hidden_size": [256, 512],
    "bidirectional": [False, True],            
    "attention": [False, True],             
    "balance_strategy": ["oversample", None] 
}

keys, values = zip(*grid_parameters.items())
experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]

best_overall_acc = 0.0
best_overall_config_name = ""

for index, updates in enumerate(experiments, 1):
    exp_id = (
        f"mid_grid_{index:02d}_"
        f"hid{updates['hidden_size']}_"
        f"bi{int(updates['bidirectional'])}_"
        f"att{int(updates['attention'])}_"
        f"bal{updates['balance_strategy']}"
    )
    
    print(f"Iteration {index}/{len(experiments)} | ID: {exp_id}")
    
    current_config = copy.deepcopy(baseline_config)
    for key, value in updates.items():
        setattr(current_config, key, value)
        
    try:
        current_acc = evaluate_cv(exp_id, current_config, training_data, num_folds=3)
        
        if isinstance(current_acc, (int, float)):
            print(f"Result: Acc = {current_acc:.4f}")
            if current_acc > best_overall_acc:
                best_overall_acc = current_acc
                best_overall_config_name = exp_id
                
    except Exception as e:
        print(f"Error w {exp_id}: {str(e)}")

if best_overall_config_name:
    print(f"Best scenario: {best_overall_config_name} (Acc: {best_overall_acc:.4f})")


Iteration 1/16 | ID: mid_grid_01_hid256_bi0_att0_baloversample
. . . mid_grid_01_hid256_bi0_att0_baloversample           acc = 0.5922 ± 0.0284
Iteration 2/16 | ID: mid_grid_02_hid256_bi0_att0_balNone
. . . mid_grid_02_hid256_bi0_att0_balNone                 acc = 0.4488 ± 0.0570
Iteration 3/16 | ID: mid_grid_03_hid256_bi0_att1_baloversample
. . . mid_grid_03_hid256_bi0_att1_baloversample           acc = 0.6162 ± 0.0224
Iteration 4/16 | ID: mid_grid_04_hid256_bi0_att1_balNone
. . . mid_grid_04_hid256_bi0_att1_balNone                 acc = 0.5092 ± 0.0645
Iteration 5/16 | ID: mid_grid_05_hid256_bi1_att0_baloversample
. . . mid_grid_05_hid256_bi1_att0_baloversample           acc = 0.6354 ± 0.0080
Iteration 6/16 | ID: mid_grid_06_hid256_bi1_att0_balNone
. . . mid_grid_06_hid256_bi1_att0_balNone                 acc = 0.5889 ± 0.0245
Iteration 7/16 | ID: mid_grid_07_hid256_bi1_att1_baloversample
. . . mid_grid_07_hid256_bi1_att1_baloversample           acc = 0.6059 ± 0.0083
Iteration 8/16 | 

KeyboardInterrupt: 

Unfortunately grid search did not manage to improve our network's accuracy. We will continue with the configuration from the previous step.

## Final Model configuration

After all experiments, we found out that the best configuration is:
NetConfig( hidden_size=512, num_layers=2, dropout=0.3, epochs=100, bidirectional=False, batch_size=32, balance_strategy="oversample", 
        attention=False, scheduler_type=None)